# Plot one trip's positions

Maps the raw vehicle positions recorded for a single trip during one `run.py` search.

Reads `outputs/<run_ts>/raw_samples/feed=<slug>/`, so the run must have had `write_raw_samples: true`.
Each row there is a position whose `vehicle_timestamp` changed since the previous poll — unchanged
repeats were deduped by the poller, so every point is a distinct update.

## Configuration

- `RUN_TS` — the search, i.e. the `outputs/` folder name. `None` picks the most recent run with raw samples.
- `FEED_NAME` — the agency's feed, exactly as written in `config.yml`.
- `TRIP_ID` — a specific trip. Leave as `None` to pick by `TRIP_INDEX` instead.
- `TRIP_INDEX` — the trip's position in the run's trip list sorted by `trip_id` (0-based, negative counts from the end).

In [ ]:
OUTPUT_DIR = "./outputs"
RUN_TS = "20261008T002947Z"
FEED_NAME = "AC Transit VehiclePositions"
TRIP_ID = None  # e.g. "554-2"; overrides TRIP_INDEX when set
TRIP_INDEX = 100
LOCAL_TZ = "America/Los_Angeles"

In [ ]:
from pathlib import Path

import branca.colormap as cm
import folium
import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString

from auth import SecretResolver
from poller import slugify

## Locate the search and feed

In [ ]:
output_dir = Path(OUTPUT_DIR)
runs_with_samples = sorted(p.parent.name for p in output_dir.glob("*/raw_samples"))
if not runs_with_samples:
    raise FileNotFoundError(f"No runs with raw_samples under {output_dir.resolve()}")

run_ts = RUN_TS or runs_with_samples[-1]
samples_dir = output_dir / run_ts / "raw_samples"
if not samples_dir.is_dir():
    raise FileNotFoundError(f"{samples_dir} not found. Runs with raw samples: {runs_with_samples}")

available_feeds = sorted(p.name.removeprefix("feed=") for p in samples_dir.glob("feed=*"))
feed_dir = samples_dir / f"feed={slugify(FEED_NAME)}"
if not feed_dir.is_dir():
    raise FileNotFoundError(f"No samples for {FEED_NAME!r} in run {run_ts}. Available feed slugs: {available_feeds}")

print(f"run:   {run_ts}")
print(f"feed:  {FEED_NAME}  ({feed_dir.name})")

## Choose the trip

In [ ]:
samples = pd.read_parquet(feed_dir)
samples = samples.dropna(subset=["trip_id", "latitude", "longitude"])

trips = (
    samples.groupby("trip_id")
    .agg(
        n_positions=("poll_seq", "size"),
        n_vehicles=("vehicle_id", "nunique"),
        first_seen=("vehicle_timestamp", "min"),
        last_seen=("vehicle_timestamp", "max"),
    )
    .sort_index()
    .reset_index()
)
if trips.empty:
    raise ValueError(f"No positions with a trip_id for {FEED_NAME!r} in run {run_ts}")

if TRIP_ID is not None:
    if TRIP_ID not in set(trips["trip_id"]):
        raise KeyError(f"trip_id {TRIP_ID!r} not in this run/feed ({len(trips)} trips available)")
    trip_id = TRIP_ID
else:
    if not -len(trips) <= TRIP_INDEX < len(trips):
        raise IndexError(f"TRIP_INDEX {TRIP_INDEX} out of range for {len(trips)} trips (0..{len(trips) - 1})")
    trip_id = trips["trip_id"].iloc[TRIP_INDEX]

print(f"{len(trips)} trips; selected trip_id = {trip_id!r}")
trips

## Build the trip's points and path

Ordered by `poll_seq`, the order the poller saw them. A trip can be served by more than one
vehicle (a mid-trip swap, or a feed reusing trip ids), so the path is drawn per vehicle.

In [ ]:
trip = samples[samples["trip_id"] == trip_id].sort_values(["vehicle_id", "poll_seq"]).reset_index(drop=True)

ts = pd.to_datetime(trip["vehicle_timestamp"].astype("float64"), unit="s", utc=True).dt.tz_convert(LOCAL_TZ)
trip["elapsed_s"] = (trip["vehicle_timestamp"] - trip["vehicle_timestamp"].min()).astype("float64")
trip["gap_s"] = trip.groupby("vehicle_id")["vehicle_timestamp"].diff().astype("float64")
# GeoJSON properties must be JSON-serialisable, so times go in as strings.
trip["time"] = ts.dt.strftime("%Y-%m-%d %H:%M:%S %Z")

points = gpd.GeoDataFrame(
    trip[["vehicle_id", "poll_seq", "time", "elapsed_s", "gap_s", "speed", "bearing", "had_absence_since_last_change", "latitude", "longitude"]],
    geometry=gpd.points_from_xy(trip["longitude"], trip["latitude"]),
    crs="EPSG:4326",
)

paths = gpd.GeoDataFrame(
    [
        {"vehicle_id": vid, "n_positions": len(g), "geometry": LineString(zip(g["longitude"], g["latitude"]))}
        for vid, g in trip.groupby("vehicle_id")
        if len(g) >= 2
    ],
    geometry="geometry",
    crs="EPSG:4326",
)

print(f"{len(points)} positions across {trip['vehicle_id'].nunique()} vehicle(s), "
      f"{trip['time'].iloc[0]} → {trip['time'].iloc[-1]}")

## Map

Points are coloured by seconds since the trip's first position. Hover for details; click a point for
its full record. Points outlined in red follow an absence — the vehicle dropped out of the feed (or a
poll failed) before this update, so the gap leading into it isn't a clean cadence measurement.

In [ ]:
colormap = cm.LinearColormap(["#2c7bb6", "#ffffbf", "#d7191c"], vmin=0, vmax=max(points["elapsed_s"].max(), 1))
colormap.caption = "Seconds since first position"

# CARTO basemaps need an API key; without one every tile is an "API KEY REQUIRED" watermark.
# Read via the same Secret Manager path as feed credentials, so run `glogin` first.
carto_key = SecretResolver().secret_value("CARTO_BASEMAPS_KEY")
m = folium.Map(tiles=None)
folium.TileLayer(
    tiles=f"https://basemaps.cartocdn.com/rastertiles/light_all/{{z}}/{{x}}/{{y}}{{r}}.png?key={carto_key}",
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>',
    name="CARTO Positron",
    max_zoom=20,
).add_to(m)
m.fit_bounds([[points["latitude"].min(), points["longitude"].min()], [points["latitude"].max(), points["longitude"].max()]])

if not paths.empty:
    folium.GeoJson(
        paths,
        name="path",
        style_function=lambda f: {"color": "#555555", "weight": 2, "opacity": 0.6},
        tooltip=folium.GeoJsonTooltip(fields=["vehicle_id", "n_positions"], aliases=["Vehicle", "Positions"]),
    ).add_to(m)

folium.GeoJson(
    points,
    name="positions",
    marker=folium.CircleMarker(radius=5, fill=True, fill_opacity=0.9, weight=1),
    style_function=lambda f: {
        "fillColor": colormap(f["properties"]["elapsed_s"]),
        "color": "#d7191c" if f["properties"]["had_absence_since_last_change"] else "#333333",
        "weight": 2.5 if f["properties"]["had_absence_since_last_change"] else 1,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["time", "vehicle_id", "gap_s", "speed"],
        aliases=["Time", "Vehicle", "Gap since previous (s)", "Speed (m/s)"],
    ),
    popup=folium.GeoJsonPopup(
        fields=["poll_seq", "time", "elapsed_s", "gap_s", "vehicle_id", "speed", "bearing", "had_absence_since_last_change", "latitude", "longitude"],
    ),
).add_to(m)

colormap.add_to(m)
folium.LayerControl().add_to(m)
m